# Partie 2 — SparkCore et RDD

Projet Écosystème Big Data 2 — FastFood.ma — EMSI 5ème année, 2026-2027.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = Path("/work")
sys.path[:0] = [str(root / "src"), str(root / "third_party")]


## Données

Le fichier d'entrée est `ORDERS_HISTORY`, produit par `producer.py --mode file --n 50000` puis copié dans `lake/bronze/history/`. Les 2 % de doublons ne sont pas retirés dans cette partie. L'écart avec le chiffre d'affaires dédoublonné de la partie 3 mesure ce biais.


In [ ]:
from fastfood.partie2 import run
from fastfood.session import build_spark

spark = build_spark("fastfood-partie2-rdd")
result = run(spark)
result["baskets"], result["top5"]


## Lineage, stages, shuffle

`textFile`, `map` et `filter` sont des transformations **étroites** : une partition produit une partition, rien ne circule entre les machines.
`reduceByKey` et `aggregateByKey` sont **larges**. Spark repartitionne par ville, écrit un shuffle, et ouvre un second stage pour réduire.

L'interface de l'application est sur le port 4040, onglet Stages. La frontière de stage correspond au shuffle : *Shuffle Write* d'un côté, *Shuffle Read* de l'autre.

Le `toDebugString` affiché au-dessus est ce lineage. Si un executor disparaît, Spark recalcule la partition perdue à partir des parents, sans fichier intermédiaire.
